# XGBoost: Gradient Boosting

## Overview
XGBoost is a gradient boosting library for high-performance machine learning. It provides:
- Fast training on large datasets
- Regularization to prevent overfitting
- Feature importance
- Cross-validation

## Why XGBoost for Engineering?
- Build accurate surrogate models
- Handle non-linear relationships
- Fast inference for real-time applications

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 1. Basic XGBoost Regression
print("=== Basic XGBoost Regression ===")

# Generate data
np.random.seed(42)
n_samples = 1000
X = np.random.randn(n_samples, 5)
y = (np.sin(X[:, 0]) + 
     X[:, 1]**2 + 
     np.exp(-X[:, 2]) + 
     X[:, 3] * X[:, 4] + 
     np.random.randn(n_samples) * 0.1)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create XGBoost model
model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8
)

# Train
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.6f}")
print(f"R²: {r2:.6f}")

# 2. Feature Importance
print("\n=== Feature Importance ===")

importances = model.feature_importances_
plt.figure(figsize=(10, 6))
plt.bar(range(len(importances)), importances)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('XGBoost Feature Importance')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cross-Validation

Use cross-validation to tune hyperparameters.

In [ ]:
# 3. Cross-Validation
print("\n=== Cross-Validation ===")

from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.2]
}

# Create grid search
grid_search = GridSearchCV(
    xgb.XGBRegressor(),
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

# Fit
grid_search.fit(X_train, y_train)

# Best parameters
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best MSE: {-grid_search.best_score_:.6f}")

# Evaluate best model
y_pred_cv = grid_search.best_estimator_.predict(X_test)
mse_cv = mean_squared_error(y_test, y_pred_cv)
r2_cv = r2_score(y_test, y_pred_cv)

print(f"\nTest MSE: {mse_cv:.6f}")
print(f"Test R²: {r2_cv:.6f}")

## Engineering Application: CFD Surrogate Model

Build a high-accuracy surrogate model for CFD predictions.

In [ ]:
# 4. Engineering Application: CFD Surrogate
print("\n=== CFD Surrogate Model ===")

# Generate synthetic CFD data
np.random.seed(42)
n_samples = 500

# Parameters: Re, alpha, M, t/c, roughness
Re = np.random.uniform(1e5, 1e6, n_samples)
alpha = np.random.uniform(0, 20, n_samples)
M = np.random.uniform(0.3, 0.9, n_samples)
tc = np.random.uniform(0.05, 0.2, n_samples)
roughness = np.random.uniform(0, 1e-6, n_samples)

# Drag coefficient (complex model)
cd = (0.02 + 
      0.01 * (Re / 1e6) + 
      0.005 * (alpha / 10)**2 + 
      0.02 * (M - 0.5)**2 +
      0.001 * (tc - 0.1)**2 +
      0.0001 * (roughness / 1e-6)**2 +
      0.001 * (Re / 1e6) * (alpha / 10) +
      np.random.randn(n_samples) * 0.001)

# Create feature matrix
X_cfd = np.column_stack([Re, alpha, M, tc, roughness])

# Split
X_train, X_test, y_train, y_test = train_test_split(X_cfd, cd, 
                                                     test_size=0.2, random_state=42)

# Train XGBoost
model_cfd = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8
)
model_cfd.fit(X_train, y_train)

# Evaluate
y_pred_cfd = model_cfd.predict(X_test)
mse_cfd = mean_squared_error(y_test, y_pred_cfd)
r2_cfd = r2_score(y_test, y_pred_cfd)

print(f"MSE: {mse_cfd:.6f}")
print(f"R²: {r2_cfd:.6f}")

# Plot predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_cfd, alpha=0.5, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('Actual Cd')
plt.ylabel('Predicted Cd')
plt.title('XGBoost CFD Surrogate: Predictions vs Actual')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Feature importance
importances = model_cfd.feature_importances_
feature_names = ['Re', 'alpha', 'M', 't/c', 'roughness']

plt.figure(figsize=(10, 6))
plt.bar(feature_names, importances)
plt.xlabel('Parameter')
plt.ylabel('Importance')
plt.title('CFD Parameter Importance')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercises

1. Compare XGBoost vs scikit-learn's GradientBoosting.
2. Use early stopping to prevent overfitting.
3. Build a surrogate model for a 10-parameter problem.